# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hussaintinwala2/Flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### My ranked action playbook

The ranked queue is a decision-support tool for prioritizing content pages for human review. Higher-ranked pages receive attention first based on observable search and content signals available at the decision point.

The ranking does not mean that a page definitely needs a refresh. It identifies pages that are more worth reviewing given the available evidence.

### Reason codes

- `STALE_AND_LOW_CTR` — the page has been updated relatively long ago and has low observed CTR.
- `LOW_CTR` — the page has low observed CTR and may deserve a review of search-result alignment.
- `STALE` — the page has been updated relatively long ago and may benefit from a freshness review.

### Action mapping

| Reason code | Suggested action |
|---|---|
| `STALE_AND_LOW_CTR` | `REVIEW_REFRESH` |
| `LOW_CTR` | `REVIEW_SEARCH_SNIPPET` |
| `STALE` | `REVIEW_REFRESH` |

The reason code explains why the page was ranked. A human reviewer must decide whether the suggested action is appropriate for the specific page.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Connect to the FlyRank warehouse

import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("DuckDB connected.")

DuckDB connected.


In [12]:
# Rebuild the baseline action queue from March 2026 decision-point data

model_data = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_march,
        SUM(f.gsc_clicks) AS clicks_march,
        AVG(f.gsc_avg_position) AS avg_position_march
    FROM {TABLES['fact_daily']} f
    WHERE f.report_date >= DATE '2026-03-01'
      AND f.report_date < DATE '2026-04-01'
    GROUP BY
        f.client_hash_id,
        f.content_hash_id
""").df()

# Calculate CTR
model_data["ctr"] = (
    model_data["clicks_march"] /
    model_data["impressions_march"].replace(0, np.nan)
)

# Recover the content update date
content_data = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        content_created_date
    FROM {TABLES['dim_content']}
    WHERE is_deleted IS NOT TRUE
""").df()

model_data = model_data.merge(
    content_data,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# March 31 is the decision point.
decision_date = pd.Timestamp("2026-03-31")

model_data["content_updated_date"] = pd.to_datetime(
    model_data["content_updated_date"],
    errors="coerce"
)

model_data["days_since_update"] = (
    decision_date - model_data["content_updated_date"]
).dt.days

# Keep usable rows
model_data = model_data.dropna(
    subset=["ctr", "days_since_update"]
).copy()

# Baseline signal thresholds
stale = model_data["days_since_update"] > 180
low_ctr = model_data["ctr"] < 0.03

# Reason codes and actions
model_data["reason_code"] = np.select(
    [
        stale & low_ctr,
        low_ctr,
        stale
    ],
    [
        "STALE_AND_LOW_CTR",
        "LOW_CTR",
        "STALE"
    ],
    default="OTHER"
)

model_data["action"] = np.select(
    [
        model_data["reason_code"] == "STALE_AND_LOW_CTR",
        model_data["reason_code"] == "LOW_CTR",
        model_data["reason_code"] == "STALE"
    ],
    [
        "REVIEW_REFRESH",
        "REVIEW_SEARCH_SNIPPET",
        "REVIEW_REFRESH"
    ],
    default="NO_ACTION"
)

# Baseline score:
# staleness contributes up to 90 points,
# low CTR contributes up to 10 points.
model_data["staleness_score"] = (
    model_data["days_since_update"].clip(lower=0, upper=365) / 365 * 90
)

model_data["ctr_score"] = (
    ((0.03 - model_data["ctr"]).clip(lower=0) / 0.03) * 10
)

model_data["priority_score"] = (
    model_data["staleness_score"] +
    model_data["ctr_score"]
)

# Keep only actionable pages
action_queue = model_data[
    model_data["reason_code"] != "OTHER"
].copy()

# Rank highest priority first
action_queue = action_queue.sort_values(
    ["priority_score", "days_since_update"],
    ascending=[False, False]
).reset_index(drop=True)

action_queue["rank"] = np.arange(1, len(action_queue) + 1)

# Final queue columns
action_queue = action_queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "priority_score",
        "reason_code",
        "action",
        "days_since_update",
        "ctr",
        "impressions_march",
        "clicks_march"
    ]
]

print("Rows in regenerated queue:", len(action_queue))
print("\nReason codes:")
print(action_queue["reason_code"].value_counts())

print("\nTop 10:")
display(action_queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in regenerated queue: 173629

Reason codes:
reason_code
LOW_CTR              173401
STALE_AND_LOW_CTR       227
STALE                     1
Name: count, dtype: int64

Top 10:


,rank,client_hash_id,content_hash_id,priority_score,reason_code,action,days_since_update,ctr,impressions_march,clicks_march
0,1,client_65de48885f4ef01b,content_38c60323fd1608ec,84.712329,STALE_AND_LOW_CTR,REVIEW_REFRESH,303.0,0.0,5.0,0.0
1,2,client_65de48885f4ef01b,content_325df589607ca257,84.712329,STALE_AND_LOW_CTR,REVIEW_REFRESH,303.0,0.0,1.0,0.0
2,3,client_65de48885f4ef01b,content_ee6f61ff4145746c,84.712329,STALE_AND_LOW_CTR,REVIEW_REFRESH,303.0,0.0,7.0,0.0
3,4,client_65de48885f4ef01b,content_19f71daba0876547,84.712329,STALE_AND_LOW_CTR,REVIEW_REFRESH,303.0,0.0,37.0,0.0
4,5,client_65de48885f4ef01b,content_f4685cd9dee88fce,84.712329,STALE_AND_LOW_CTR,REVIEW_REFRESH,303.0,0.0,1.0,0.0
5,6,client_65de48885f4ef01b,content_a2e85700106a33b8,84.712329,STALE_AND_LOW_CTR,REVIEW_REFRESH,303.0,0.0,9.0,0.0
6,7,client_65de48885f4ef01b,content_cdd557d042d8a774,84.712329,STALE_AND_LOW_CTR,REVIEW_REFRESH,303.0,0.0,19.0,0.0
7,8,client_65de48885f4ef01b,content_3ed48fd591b26655,84.712329,STALE_AND_LOW_CTR,REVIEW_REFRESH,303.0,0.0,1.0,0.0
8,9,client_65de48885f4ef01b,content_26d4238346178145,84.712329,STALE_AND_LOW_CTR,REVIEW_REFRESH,303.0,0.0,2.0,0.0
9,10,client_65de48885f4ef01b,content_27705ab6c4138e8e,84.712329,STALE_AND_LOW_CTR,REVIEW_REFRESH,303.0,0.0,9.0,0.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended to help a human content reviewer prioritize pages for review. It is decision-support rather than an automatic content optimization system.

The queue can be used to:

- identify pages with observed low CTR or staleness signals;
- prioritize which pages should be reviewed first;
- provide a consistent reason code for why a page entered the queue;
- support discussion of whether a refresh or search-result review is worthwhile.

The ranking should be interpreted directionally. The Week-6 client-grouped validation produced a ROC-AUC of 0.538, so the learned signal is limited and should not be treated as a reliable prediction of future decline.

### Limits

The queue does not establish that a page will decline, that updating it will improve performance, or that the recommended action is optimal.

The ranking is based on the available decision-point signals and does not capture all factors that can affect content performance, such as changes in search demand, competitors, search-engine behavior, content quality, or business priorities.

The queue should therefore be reviewed by a person before any content change is made.

### Cost/value thinking

Human review should start with the highest-ranked pages because reviewer time is limited. A low-cost review may be worthwhile when a page has meaningful search exposure and a plausible issue that can be addressed. Pages with very little observed search activity may have lower expected value even when they receive a high score.

The score is therefore a prioritization aid, not a direct estimate of financial value or expected traffic gain.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the final ranked action queue
# Decision point: March 31, 2026

import numpy as np
import pandas as pd

DECISION_DATE = pd.Timestamp("2026-03-31")

# Work from the existing model_data
queue_data = model_data.copy()

# Calculate days since last update
queue_data["days_since_update"] = (
    DECISION_DATE - pd.to_datetime(
        queue_data["content_updated_date"],
        errors="coerce"
    )
).dt.days

# Remove rows whose update date was after the decision point
queue_data = queue_data[
    queue_data["days_since_update"].notna()
    & (queue_data["days_since_update"] >= 0)
].copy()

# Calculate CTR
queue_data["ctr"] = (
    queue_data["clicks_march"]
    / queue_data["impressions_march"].replace(0, np.nan)
).fillna(0)

# Signal definitions
queue_data["is_low_ctr"] = queue_data["ctr"] < 0.03
queue_data["is_stale"] = queue_data["days_since_update"] > 180

# Reason code
queue_data["reason_code"] = np.select(
    [
        queue_data["is_stale"] & queue_data["is_low_ctr"],
        queue_data["is_stale"],
        queue_data["is_low_ctr"]
    ],
    [
        "STALE_AND_LOW_CTR",
        "STALE",
        "LOW_CTR"
    ],
    default="OTHER"
)

# Action label
queue_data["action"] = np.select(
    [
        queue_data["reason_code"].isin(
            ["STALE_AND_LOW_CTR", "STALE"]
        ),
        queue_data["reason_code"] == "LOW_CTR"
    ],
    [
        "REVIEW_REFRESH",
        "REVIEW_SEARCH_SNIPPET"
    ],
    default="NO_ACTION"
)

# Priority score
queue_data["priority_score"] = (
    queue_data["is_low_ctr"].astype(float) * 10
    + queue_data["is_stale"].astype(float) * 10
)

queue_data["priority_score"] = np.where(
    queue_data["reason_code"] == "STALE_AND_LOW_CTR",
    queue_data["priority_score"]
    + np.minimum(queue_data["days_since_update"] / 10, 65),
    queue_data["priority_score"]
)

# Keep actionable pages only
action_queue = queue_data[
    queue_data["reason_code"] != "OTHER"
].copy()

# Rank
action_queue = action_queue.sort_values(
    by=["priority_score", "days_since_update", "ctr"],
    ascending=[False, False, True]
).reset_index(drop=True)

action_queue["rank"] = np.arange(1, len(action_queue) + 1)

print("Rows in ranked queue:", len(action_queue))
print("\nReason codes:")
print(action_queue["reason_code"].value_counts())

print("\nTop 10:")
display(
    action_queue[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "priority_score",
            "reason_code",
            "action",
            "days_since_update",
            "ctr",
            "impressions_march",
            "clicks_march"
        ]
    ].head(10)
)

Rows in ranked queue: 27534

Reason codes:
reason_code
LOW_CTR              27306
STALE_AND_LOW_CTR      227
STALE                    1
Name: count, dtype: int64

Top 10:


,rank,client_hash_id,content_hash_id,priority_score,reason_code,action,days_since_update,ctr,impressions_march,clicks_march
0,1,client_65de48885f4ef01b,content_38c60323fd1608ec,50.3,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,5.0,0.0
1,2,client_65de48885f4ef01b,content_325df589607ca257,50.3,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
2,3,client_65de48885f4ef01b,content_ee6f61ff4145746c,50.3,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,7.0,0.0
3,4,client_65de48885f4ef01b,content_19f71daba0876547,50.3,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,37.0,0.0
4,5,client_65de48885f4ef01b,content_f4685cd9dee88fce,50.3,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
5,6,client_65de48885f4ef01b,content_a2e85700106a33b8,50.3,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,9.0,0.0
6,7,client_65de48885f4ef01b,content_cdd557d042d8a774,50.3,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,19.0,0.0
7,8,client_65de48885f4ef01b,content_3ed48fd591b26655,50.3,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
8,9,client_65de48885f4ef01b,content_26d4238346178145,50.3,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,2.0,0.0
9,10,client_65de48885f4ef01b,content_27705ab6c4138e8e,50.3,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,9.0,0.0


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

Every queued page requires human review before any content change is made. The queue identifies pages worth investigating; it does not decide what change should be made.

### Human review rules

Before acting on a queued page, the reviewer should:

1. Check the page's current search-result presentation and confirm that the low CTR signal is still relevant.
2. Check whether the page is actually stale and whether its topic or information has changed since the last update.
3. Consider search intent and whether the page is targeting the appropriate query or audience.
4. Check for important business, editorial, or technical context that is not represented in the queue.
5. Decide whether the suggested action is appropriate, needs modification, or should be rejected.
6. Record the reason for the final decision so that the queue remains decision-support rather than automatic optimization.

### No-go list

The system should **not** automatically:

- rewrite or publish content;
- change titles, descriptions, or other search-result elements without review;
- delete, redirect, or substantially restructure a page;
- make claims about guaranteed traffic or ranking improvement;
- override editorial, legal, brand, or business requirements;
- treat a high priority score as proof that a page is underperforming for a specific cause;
- make decisions where the available signals are missing, clearly inconsistent, or outside the validated decision-point window.

The final action remains with a human reviewer. The queue is intended to reduce review effort and provide consistent prioritization, not to replace content judgment.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Human review + no-go checks
# Verify that the queue contains only pages requiring human review
# and that the actions remain review-oriented rather than automatic.

print("=== HUMAN REVIEW / NO-GO CHECK ===")

print(f"Actionable pages requiring review: {len(action_queue):,}")

print("\nActions in queue:")
print(
    action_queue["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="n")
)

print("\nReason codes in queue:")
print(
    action_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

# Check that all actions are review actions.
review_actions = {
    "REVIEW_REFRESH",
    "REVIEW_SEARCH_SNIPPET"
}

unexpected_actions = set(action_queue["action"].dropna()) - review_actions

print("\nNo-go automation check:")
if not unexpected_actions:
    print("PASS: all queued actions require human review.")
else:
    print("WARNING: unexpected non-review actions found:", unexpected_actions)

# Check that no rows are marked as automatic actions.
automatic_keywords = ["AUTO", "PUBLISH", "DELETE", "REDIRECT", "REWRITE"]

automatic_actions = [
    action for action in action_queue["action"].dropna().unique()
    if any(keyword in action.upper() for keyword in automatic_keywords)
]

if not automatic_actions:
    print("PASS: no automatic publish/delete/redirect/rewrite action is present.")
else:
    print("WARNING: potentially automatic actions found:", automatic_actions)

=== HUMAN REVIEW / NO-GO CHECK ===
Actionable pages requiring review: 27,534

Actions in queue:
                  action      n
0  REVIEW_SEARCH_SNIPPET  27306
1         REVIEW_REFRESH    228

Reason codes in queue:
         reason_code      n
0            LOW_CTR  27306
1  STALE_AND_LOW_CTR    227
2              STALE      1

No-go automation check:
PASS: all queued actions require human review.
PASS: no automatic publish/delete/redirect/rewrite action is present.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / retrain triggers

The playbook should be monitored because search behavior, content freshness, and the usefulness of the signals can change over time.

### Monitoring

The following should be checked periodically:

- the distribution of reason codes and actions;
- CTR and freshness distributions in the decision-point data;
- the number of pages entering the queue;
- whether the highest-priority pages still show meaningful review issues;
- whether reviewers frequently reject or override the suggested action.

### Retrain / review triggers

The model or rule should be reviewed if:

- the observed feature distributions change substantially;
- the queue size changes sharply without an obvious explanation;
- the relationship between the signals and observed outcomes weakens;
- grouped validation performance falls materially below the previously observed ROC-AUC of 0.538;
- human reviewers consistently disagree with the recommended actions.

A trigger should start an investigation rather than automatically retraining or changing the playbook. Any retraining should use a newly defined decision-point window and repeat the leakage and grouped-validation checks.

The monitoring process is intended to identify when the current decision-support workflow may have become stale; it does not establish that a model update will improve outcomes.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Monitoring / retrain trigger checks

print("=== MONITORING / RETRAIN TRIGGER CHECK ===")

print(f"Current actionable queue size: {len(action_queue):,}")

print("\nReason-code distribution:")
print(
    action_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

print("\nAction distribution:")
print(
    action_queue["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="n")
)

print("\nPriority-score summary:")
print(action_queue["priority_score"].describe())

print("\nCTR summary:")
print(action_queue["ctr"].describe())

print("\nDays-since-update summary:")
print(action_queue["days_since_update"].describe())

print("\nPreviously observed honest validation ROC-AUC: 0.538")
print("Monitoring note: a material decline from this reference should trigger review.")
print("No automatic retraining is triggered by this notebook.")

=== MONITORING / RETRAIN TRIGGER CHECK ===
Current actionable queue size: 27,534

Reason-code distribution:
         reason_code      n
0            LOW_CTR  27306
1  STALE_AND_LOW_CTR    227
2              STALE      1

Action distribution:
                  action      n
0  REVIEW_SEARCH_SNIPPET  27306
1         REVIEW_REFRESH    228

Priority-score summary:
count    27534.000000
mean        10.267168
std          2.945724
min         10.000000
25%         10.000000
50%         10.000000
75%         10.000000
max         50.300000
Name: priority_score, dtype: float64

CTR summary:
count    27534.000000
mean         0.001477
std          0.004418
min          0.000000
25%          0.000000
50%          0.000000
75%          0.001567
max          0.500000
Name: ctr, dtype: float64

Days-since-update summary:
count    27534.000000
mean        40.182974
std         27.382973
min          7.000000
25%         34.000000
50%         34.000000
75%         34.000000
max        303.000000
Name

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 5. Exports for the paper

The ranked action queue is exported from this notebook so that the paper can trace its recommendations back to the same decision-support workflow.

The queue is regenerated from the decision-point data and contains the ranked actions, reason codes, priority scores, and supporting signals used for prioritization.

The CSV is treated as a generated data artifact and is not committed to git. The notebook remains the reproducible source for regenerating it.

The exported queue should be interpreted as a human-review prioritization list, not as a production decision system or a prediction of guaranteed content performance.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 5: Export the ranked queue for the paper

from pathlib import Path

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QUEUE_PATH = OUTPUT_DIR / "baseline_action_playbook_queue.csv"

export_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "priority_score",
    "reason_code",
    "action",
    "days_since_update",
    "ctr",
    "impressions_march",
    "clicks_march"
]

# Export only columns that exist in the final queue.
export_cols = [col for col in export_cols if col in action_queue.columns]

action_queue[export_cols].to_csv(
    QUEUE_PATH,
    index=False
)

print("=== PAPER EXPORT ===")
print(f"Queue rows exported: {len(action_queue):,}")
print(f"Export path: {QUEUE_PATH}")
print(f"Columns exported: {len(export_cols)}")

print("\nExport check:")
print(f"File exists: {QUEUE_PATH.exists()}")

# Verify that the exported file can be read back.
export_check = pd.read_csv(QUEUE_PATH)

print(f"Rows read back: {len(export_check):,}")

if len(export_check) == len(action_queue):
    print("PASS: exported queue row count matches the notebook queue.")
else:
    print("WARNING: exported row count does not match the notebook queue.")

=== PAPER EXPORT ===
Queue rows exported: 27,534
Export path: work/outputs/baseline_action_playbook_queue.csv
Columns exported: 10

Export check:
File exists: True
Rows read back: 27,534
PASS: exported queue row count matches the notebook queue.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.